# Utility of Application

One problem this system can solve is the automatic detection of misaligned bottles in packaging and robotic handling processes. In production lines or automated stations, it is crucial that bottles are in the correct position, typically vertical, so they can be filled, labeled, transported, or picked up by a robotic arm without errors. If a bottle is lying down, tilted, or partially out of position, it can cause packaging failures, line jams, gripping errors, or process interruptions. In this context, a system based on instance segmentation is useful because it not only detects the presence of the bottle but also analyzes its shape and actual orientation, allowing the system to determine whether the object is ready to continue in the automated flow or if it requires correction.

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

# Cargar modelo de segmentación
model = YOLO("yolov8n-seg.pt") 

cap = cv2.VideoCapture(0)

TARGET_CLASSES = ["bottle"]
MIN_MASK_AREA = 1800  

def classify_object_position(label, binary_mask, frame_shape):
    h_img, w_img = frame_shape[:2]
    
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return "UNKNOWN", None, None

    largest_contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(largest_contour)

    if area < MIN_MASK_AREA:
        return "TOO SMALL", largest_contour, None

    # Bounding box normal para dibujo
    x, y, wb, hb = cv2.boundingRect(largest_contour)

    # RECTÁNGULO ROTADO: Ignora si la botella está inclinada
    rect = cv2.minAreaRect(largest_contour)
    # Usamos _ para las variables que no necesitamos (centro y ángulo)
    _, (w_rot, h_rot), _ = rect

    # Comparamos las dimensiones reales del objeto:
    
    # Identificamos cuál es el lado más largo y el más corto de la botella
    lado_largo = max(w_rot, h_rot)
    lado_corto = min(w_rot, h_rot)
    
    # El Aspect Ratio real 
    aspect_ratio = lado_corto / lado_largo if lado_largo != 0 else 0

    # Determinar si está horizontal o vertical basándonos en el Bounding Box NORMAL
    # Pero ahora validamos que sea una forma alargada (AR < 0.7)
    if hb > wb:
        orientation = "vertical"
    else:
        orientation = "horizontal"
        
    status = "CORRECT POSITION" if orientation == "vertical" else "FALLEN / MISPLACED"

    # VALIDACIÓN DE BORDES
    touches_edge = x <= 5 or (x + wb) >= w_img - 5 or (y + hb) >= h_img - 5
    if touches_edge and area < 4000:
        status = "PARTIAL / UNCERTAIN"

    return status, largest_contour, (x, y, wb, hb, aspect_ratio)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Realizar inferencia
    results = model(frame, verbose=False) 
    result = results[0]

    annotated = frame.copy()
    h, w = frame.shape[:2]

    if result.masks is not None and result.boxes is not None:
        masks = result.masks.data.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()

        for i, mask in enumerate(masks):
            class_id = int(classes[i])
            label = model.names[class_id]
            conf = confs[i]

            if label in TARGET_CLASSES and conf > 0.3:
                # Redimensionar máscara al tamaño original del frame
                mask_resized = cv2.resize(mask, (w, h))
                binary_mask = (mask_resized > 0.5).astype(np.uint8)

                # Clasificar posición
                status, contour, info = classify_object_position(label, binary_mask, (h, w))

                if contour is not None and info is not None:
                    x, y, bw, bh, ar = info

                    # Definir colores según el estado
                    if "CORRECT" in status:
                        color = (0, 255, 0) # Verde
                    elif "FALLEN" in status:
                        color = (0, 0, 255) # Rojo
                    else:
                        color = (0, 255, 255) # Amarillo

                    # Dibujar transparencia de la máscara
                    overlay = annotated.copy()
                    overlay[binary_mask == 1] = color
                    annotated = cv2.addWeighted(overlay, 0.3, annotated, 0.7, 0)

                    # Dibujar contorno y cuadro
                    cv2.drawContours(annotated, [contour], -1, (255, 255, 255), 1)
                    cv2.rectangle(annotated, (x, y), (x + bw, y + bh), color, 2)

                    # Etiquetas de texto
                    cv2.putText(annotated, f"{label}: {status}", (x, y - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                    cv2.putText(annotated, f"AR: {ar:.2f} Conf: {conf:.2f}", (x, y + bh + 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # Mostrar resultado
    cv2.imshow("Object Position Detector", annotated)

    # Salir con 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()